In [101]:
import pandas as pd
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df=pd.read_csv(url)
display(df.head())
print(df.shape)




,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


(891, 12)


1.¿Cuántos pasajeros hay en el dataset y cuántos PassengerId distintos aparecen? ¿Coinciden los dos resultados?

In [102]:
total_registros =len(df)
indentificadores_distintos=df["PassengerId"].nunique()

print ("Total de pasajeros: ",total_registros)
print("PassengerId distintos:", indentificadores_distintos)
print("¿Coinciden?", total_registros == indentificadores_distintos)

Total de pasajeros:  891
PassengerId distintos: 891
¿Coinciden? True


¿Cuántos pasajeros de cada clase aparecen en el archivo? ¿Qué clase tiene más pasajeros registrados?

In [103]:
print (df["Pclass"].value_counts().sort_index())
tabla_clases = (
    df["Pclass"]
    .value_counts()
    .sort_index()
    .rename_axis("Clase")
    .reset_index(name="Cantidad de pasajeros")
)

display(tabla_clases)

print("La clase con mayor pasajeros fue: ",df["Pclass"].value_counts().idxmax())

Pclass
1    216
2    184
3    491
Name: count, dtype: int64


,Clase,Cantidad de pasajeros
0,1,216
1,2,184
2,3,491


La clase con mayor pasajeros fue:  3


¿Cuántos pasajeros embarcaron en cada puerto y cuántos tienen el puerto sin registrar?

In [104]:
tabla_puertos = (
    df["Embarked"]
    .value_counts(dropna=False)
    .rename_axis("Puerto de embarque")
    .reset_index(name="Cantidad de pasajeros")
)

display(tabla_puertos)

,Puerto de embarque,Cantidad de pasajeros
0,S,644
1,C,168
2,Q,77
3,NaN,2


¿Cuál es la tarifa promedio, mínima y máxima registrada para cada clase?

In [105]:
tabla_tarifas = df.groupby("Pclass")["Fare"].agg(
    Promedio="mean",
    Minimo="min",
    Maximo="max"
).reset_index()
display(tabla_tarifas)

,Pclass,Promedio,Minimo,Maximo
0,1,84.154687,0.0,512.3292
1,2,20.662183,0.0,73.5000
2,3,13.675550,0.0,69.5500


¿Cuántos pasajeros de cada clase embarcaron en cada puerto?

In [106]:
tabla_clase_puerto = pd.crosstab(
    df["Pclass"],
    df["Embarked"].fillna("Sin registrar")
)

display(tabla_clase_puerto)

Embarked,C,Q,S,Sin registrar
Pclass,,,,
1,85,2,127,2
2,17,3,164,0
3,66,72,353,0


¿Cuántos pasajeros sobrevivieron y cuántos fallecieron en cada clase?

In [107]:
tabla_supervivencia = pd.crosstab(
    df["Pclass"],
    df["Survived"]
)

tabla_supervivencia = tabla_supervivencia.rename(
    columns={
        0: "Fallecieron",
        1: "Sobrevivieron"
    }
)

display(tabla_supervivencia)

Survived,Fallecieron,Sobrevivieron
Pclass,,
1,80,136
2,97,87
3,372,119


¿Cuántos pasajeros viajaban con cero, uno o dos o más familiares registrados a bordo?

In [108]:
familiares = df["SibSp"] + df["Parch"]
categoria_familiares = pd.Series(index=df.index, dtype="object")
categoria_familiares.loc[familiares == 0] = "Cero"

categoria_familiares.loc[familiares == 1] = "Uno"

categoria_familiares.loc[familiares >= 2] = "Dos o más"

categoria_familiares.loc[familiares.isna()] = "Sin dato"
tabla_familiares = (
    categoria_familiares
    .value_counts()
    .rename_axis("Familiares a bordo")
    .reset_index(name="Cantidad de pasajeros")
)

display(tabla_familiares)

,Familiares a bordo,Cantidad de pasajeros
0,Cero,537
1,Dos o más,193
2,Uno,161


¿Cuántos pasajeros tenían menos de 18 años, cuántos tenían 18 o más y cuántos tenían la edad sin registrar? Desglose el resultado por clase.

In [109]:
df["Age"] < 18
categoria_edad = pd.Series(
    "Sin dato",
    index=df.index
)

categoria_edad.loc[df["Age"] < 18] = "Menores de 18"

categoria_edad.loc[df["Age"] >= 18] = "18 o más"


tabla_edades = pd.crosstab(
    df["Pclass"],
    categoria_edad
)

tabla_edades = tabla_edades.reindex(
    columns=["Menores de 18", "18 o más", "Sin dato"],
    fill_value=0
)

tabla_edades["Total"] = tabla_edades.sum(axis=1)

display(tabla_edades)


col_0,Menores de 18,18 o más,Sin dato,Total
Pclass,,,,
1,12,174,30,216
2,23,150,11,184
3,78,277,136,491


¿Cuántos códigos de boleto distintos hay? ¿Cuántos aparecen asociados a más de un pasajero y cuál aparece en más registros?

In [110]:

codigos_distintos = df["Ticket"].nunique()
conteo_tickets = df["Ticket"].value_counts()
tickets_repetidos = (conteo_tickets > 1).sum()
ticket_mas_frecuente = conteo_tickets.idxmax()
frecuencia_maxima = conteo_tickets.max()

print("Cantidad de códigos distintos:", codigos_distintos)
print("Códigos asociados a más de un pasajero:", tickets_repetidos)
print("Código más frecuente:", ticket_mas_frecuente)
print("Cantidad de pasajeros asociados:", frecuencia_maxima)

tickets_mas_frecuentes = conteo_tickets[
    conteo_tickets == frecuencia_maxima
]

display(tickets_mas_frecuentes)



Cantidad de códigos distintos: 681
Códigos asociados a más de un pasajero: 134
Código más frecuente: 347082
Cantidad de pasajeros asociados: 7


Ticket
347082      7
1601        7
CA. 2343    7
Name: count, dtype: int64

¿En qué clase hay más registros sin cabina? Para cada clase, muestren cuántos registros tienen cabina, cuántos carecen de ella y cuántos hay en total.

In [111]:

tabla_cabinas = df.groupby("Pclass").agg(
    Con_cabina=("Cabin", lambda x: x.notna().sum()),
    Sin_cabina=("Cabin", lambda x: x.isna().sum()),
    Total=("Cabin", "size")
).reset_index()

display(tabla_cabinas)

clase_mas_sin_cabina = tabla_cabinas.loc[
    tabla_cabinas["Sin_cabina"].idxmax()
]

print("Clase con más registros sin cabina:")
display(clase_mas_sin_cabina)


,Pclass,Con_cabina,Sin_cabina,Total
0,1,176,40,216
1,2,16,168,184
2,3,12,479,491


Clase con más registros sin cabina:


Pclass          3
Con_cabina     12
Sin_cabina    479
Total         491
Name: 2, dtype: int64